#**🧠 Multi-Agent System for Smart Query Handling with Tool-Based Reasoning**
This notebook demonstrates a modular Conversational AI system powered by agno.agent, where each agent is equipped with a specialized tool. These agents collaborate to handle a variety of user queries like weather updates, arithmetic calculations, factual lookups, and web searches — all within a unified pipeline.

🌟 What You'll Learn
- How to build individual tool-based agents using agno.agent

- How to combine them into a multi-agent team that handles diverse queries

- How to log and structure the full conversation history with role separation (user, assistant, tool)

How to evaluate the quality of each agent's performance using the LlumoClient and defined metrics



# ⚙️ **Step 1: Install Dependencies**


In [ ]:
!pip install agno duckduckgo-search wikipedia -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 953.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.8/802.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 10.6 MB/s eta 0:00:00


# 📂 **Step 2: Import Required Libraries**


In [ ]:
import os
import getpass
from agno.agent import Agent
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.wikipedia import WikipediaTools



# 🔐 **Step 3: Enter Llumo API Key Securely**



In [ ]:

# Prompt user to input their API key securely (won't be visible in output)
os.environ["LLUMO_API_KEY"] = getpass.getpass("Enter your Llumo API key: ")
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your open API key: ")

# Retrieve the API key
llumo_key = os.environ["LLUMO_API_KEY"]
openai_key = os.environ["OPENAI_API_KEY"]

Enter your Llumo API key: ··········
Enter your open API key: ··········


# **STEP 4:🔧 Setup: Defining the Agno Tools 🧑‍💻**


In [ ]:


# -------------------------------
# 🛠️ Custom Tool 1: get_weather
# -------------------------------
def get_weather(city: str) -> str:
    return (
        f"Weather in {city}:\n"
        "Day 1: Sunny, 26°C\n"
        "Day 2: Cloudy, 24°C\n"
        "Day 3: Light rain, 22°C"
    )

# -------------------------------
# 🛠️ Custom Tool 2: simple_calculator
# -------------------------------
def simple_calculator(expression: str) -> str:
    try:
        result = eval(expression)
        return f"The result of '{expression}' is {result}."
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"




# **STEP 5: Wrapping into Agno Agents and Creating a team**

In [ ]:
# Custom tool
weatherAgent = Agent(
    tools=[get_weather],
    name="WeatherAgent",
    description="Provides a 3-day weather forecast for any given city.",
    show_tool_calls=True
)
# Custom tool
calculatorAgent = Agent(
    tools=[simple_calculator],
    name="CalculatorAgent",
    description="Performs basic arithmetic calculations like addition, subtraction, etc.",
    show_tool_calls=True
)

# Predefined Tool
wikipediaAgent = Agent(
    tools=[WikipediaTools()],
    name="WikipediaAgent",
    description="Fetches summaries from Wikipedia for general knowledge and topics.",
    show_tool_calls=True
)


# Predefined Tool
searchAgent = Agent(
    tools=[DuckDuckGoTools()],
    name="SearchAgent",
    description="Performs real-time web searches using DuckDuckGo.",
    show_tool_calls=True
)

# Final Combined Agent

members = [weatherAgent, calculatorAgent, wikipediaAgent, searchAgent]
agent_team = Agent(
    team = members,
    show_tool_calls=True,
    markdown=True,
    add_history_to_messages=False
)


#**STEP 6:🛠️ Running the Agent & Creating Message History Chunks:🏃**
Let’s process the queries through the Agno agent to get responses. The results will help us evaluate the agent's performance.



### Function To Create Message Chunks

In [ ]:
def chunk_conversation(messages):
    chunks = []
    for message in messages:
        if message.role == "system":
            continue # Ignore system messages

        if message.role == "tool":
             message_obj = {
            "role": "tool",
            "tool_name": message.tool_name.replace("transfer_task_to_",""),
            "parts": {"text": message.content},
        }
        if message.role == "user":
          message_obj = {
              "role": "user",
              "parts": {"text": message.content}
          }

        if message.role == "assistant":
          message_obj = {
              "role": "assistant",
              "parts": {"text": message.content}
          }


        chunks.append(message_obj)
    return chunks

### Run Sample Queries Through the Agent Team and Capture Conversation History

In [ ]:
# --- Example Query ---
import time

queries = [
    "Hi, I'm Ravi.",
    "What's the weather forecast for Chennai?",
    "Can you calculate 18 / (3 + 3)?",
    "Show me the latest news about electric vehicles.",
    "Tell me about the history of the Eiffel Tower."
]

history = []
for question in queries:
  time.sleep(3)
  res = agent_team.run(question)
  history.append(chunk_conversation(res.messages))


INFO Setting default model to OpenAI Chat                                       
INFO Searching wikipedia for: Eiffel Tower construction history                 
INFO Searching wikipedia for: Eiffel Tower history and significance             
INFO Searching wikipedia for: Eiffel Tower historical overview                  
INFO Searching wikipedia for: Eiffel Tower renovations and events               


### Preparing Input DataFrame

In [ ]:
import time

# --- Example Queries ---
queries = [
    "Hi, I'm Ravi.",
    "What's the weather forecast for Chennai?",
    "Can you calculate 18 / (3 + 3)?",
    "Show me the latest news about electric vehicles.",
    "Tell me about the history of the Eiffel Tower."
]

results = []  # List to store the output in dict format

tools = {agent.name.lower(): agent.description for agent in members}  # Important: agent names in lowercase

# Run each query, collect result as dict
for question in queries:
    time.sleep(3)  # Delay to avoid hitting rate limits if applicable
    res = agent_team.run(question)

    history = chunk_conversation(res.messages)

    result = {
        "query": question,
        "output": history[-1]["parts"]["text"],  # Latest/last assistant reply
        "messageHistory": history,
        "tools": tools
    }
    results.append(result)



# **STEP: 7 🧠 Evaluate Agent Responses using LlumoClient**
- Uses the llumo Python library to evaluate LLM-generated responses
- LlumoClient is used to score responses on criteria such as:
  - Tool usage correctness
  - Overall quality, completeness and correctness
Helps analyze and benchmark the performance of the conversational agent

###**STEP 1: ⬇ Import necessary modules**

In [ ]:
!pip install llumo -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 3.2 MB/s eta 0:00:00


### **STEP 2: Evaluation**

In [ ]:

from llumo import LlumoClient
from llumo.functionCalling import LlumoAgent

# 🔑 Initialize the LlumoClient with your LLUMO API key
client = LlumoClient(api_key=llumo_key)  # Replace

# ✅ Use the client to evaluate agent responses using the collected DataFrame
evals =  ['Tool Reliability', 'Stepwise Progression', 'Tool Selection Accuracy', 'Final Task Alignment']
resultDf = client.evaluateAgentResponses(data = results,evals = evals)


======= Running evaluation for: Tool Reliability =======

======= Running evaluation for: Stepwise Progression =======

======= Running evaluation for: Tool Selection Accuracy =======

======= Running evaluation for: Final Task Alignment =======


### **STEP 3: Result DataFrame**

In [ ]:
resultDf

,query,output,messageHistory,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Final Task Alignment,Final Task Alignment Reason
0,"Hi, I'm Ravi.","Hello, Ravi! How can I assist you today?","[{'role': 'user', 'parts': {'text': ""Hi, I'm R...",{'weatheragent': 'Provides a 3-day weather for...,1,No tools were used in the conversation. The m...,1,No tools were used. The user query only conta...,99,"No tools were expected, and no tools were used...",100,No actual query was raised by the user. The as...
1,What's the weather forecast for Chennai?,Here's the 3-day weather forecast for Chennai:...,"[{'role': 'user', 'parts': {'text': ""What's th...",{'weatheragent': 'Provides a 3-day weather for...,99,The `weatheragent` tool successfully executed ...,99,The user query seeks a weather forecast for Ch...,99,The user requested a weather forecast. Only th...,99,The final response directly answers the user's...
2,Can you calculate 18 / (3 + 3)?,The result of the expression \( \frac{18}{(3 +...,"[{'role': 'user', 'parts': {'text': 'Can you c...",{'weatheragent': 'Provides a 3-day weather for...,100,The calculator tool successfully performed the...,99,The tool 'calculatoragent' is relevant to the ...,99,The user requested a calculation. Only the cal...,100,The assistant correctly calculated the express...
3,Show me the latest news about electric vehicles.,Here are some of the latest news articles abou...,"[{'role': 'user', 'parts': {'text': 'Show me t...",{'weatheragent': 'Provides a 3-day weather for...,99,The `searchagent` tool successfully retrieved ...,99,The user requested news about electric vehicle...,100,The user requested news about electric vehicle...,99,The assistant successfully provided a summary ...
4,Tell me about the history of the Eiffel Tower.,"The Eiffel Tower, an iconic wrought-iron latti...","[{'role': 'user', 'parts': {'text': 'Tell me a...",{'weatheragent': 'Provides a 3-day weather for...,100,The Wikipedia tool successfully retrieved and ...,100,"The tool used, wikipediaagent, is directly rel...",100,"The assistant used only the wikipediaagent, wh...",99,The assistant's final response provides a deta...
